# Unique peptides for internal and N-terminal isoforms
We will do an in-silico tryptic digestion with all the isoforms that were classified as not C-terminal in the previous notebook. The peptides will then be filtered by length (7 to 52 amino acids) and then we will sequentially add them to the database that contains the canonical sequences and the C-terminal isoforms for which we found unique peptides.

# Setup
## Import and install Packages

In [ ]:
import sys
import os
import session_info

# Add the '0_functions' folder to sys.path
sys.path.append(os.path.join(os.getcwd(), '..', '00_functions'))

In [ ]:
import pandas as pd
import re
from Bio import SeqIO
from functions import read_fastafile

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/02_Isoform_Databse"):
    print("WARNING: The working directory is not set to the '02_02_Isoform_Databse'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '02_02_Isoform_Databse' folder (\"{cwd}\").")

In [ ]:
# Data directories
input_dir = "data/digestion_input"
output_dir = "data/digestion_output"
fasta_dir = "../01_UniProt/data/raw_data/fasta"

# Tryptic in silico digestion of internal and N-terminal isoforms
## Create fasta with only non C-terminal isoforms

In [ ]:
fasta_no_duplicates = os.path.join(input_dir, "fasta_no_duplicates.fasta")
df_others = pd.read_csv("data/df_others.csv")

In [ ]:
# Convert df_others IDs to a set for fast lookup 
other_ids = set(df_others['Isoform_ID'])

filtered_records = []
# Load unique FASTA
for record in SeqIO.parse(fasta_no_duplicates, "fasta"):
    # Extract the ID from the header (handles UniProt style: sp|ID|NAME)
    header_id = record.id.split('|')[1] if '|' in record.id else record.id
    
    if header_id in other_ids:
        filtered_records.append(record)

# Save the subset for digestion
SeqIO.write(filtered_records, "data/digestion_input/isoforms_internal_nterm.fasta", "fasta")
print(f"Saved {len(filtered_records)} sequences for digestion.")

## Performing digestion with Rapid Peptides generator
Rapid Peptides Generator (RPG), is a standalone software dedicated to predict proteases-induced cleavage sites on sequences.

RPG is a python tool taking a (multi-)fasta/fastq file of proteins as input and digest each of them. We will use this tool to digest the full proteome using trypsin. We will adjust the cleavage rules to match the ones in MaxQuant. This either means Trypsin cleaves after every L and K or it cleaves after every L and K except if it is followed by P.

The resulting peptides contain information about positions of cleavage site, peptide sequences, length, mass as well as an estimation of isoelectric point (pI) of each peptide. Results are outputted in multi-fasta, CSV or TSV file.

I have defined two new enzymes. MaxQuant_Trypsin cleaves after every R and K and MaxQuant_Trypsin_P cleaves after every R and K except if they are followed by P.

In [ ]:
digestion_isoforms_Trypsin = read_fastafile(os.path.join(output_dir, 'digestion_isoforms_Trypsin.fasta'))
digestion_isoforms_Trypsin_P = read_fastafile(os.path.join(output_dir, 'digestion_isoforms_Trypsin_P.fasta'))

In [ ]:
# Filter to only contain fragments of legth between 7 and 52 amino acids
digestion_isoforms_Trypsin_filtered = digestion_isoforms_Trypsin[digestion_isoforms_Trypsin["len"] >= 7]
digestion_isoforms_Trypsin_filtered = digestion_isoforms_Trypsin_filtered[digestion_isoforms_Trypsin_filtered["len"] <= 52]

digestion_isoforms_TrypsinP_filtered = digestion_isoforms_Trypsin_P[digestion_isoforms_Trypsin_P["len"] >= 7]
digestion_isoforms_TrypsinP_filtered = digestion_isoforms_TrypsinP_filtered[digestion_isoforms_TrypsinP_filtered["len"] <= 52]

In [ ]:
#save filtered dfs as csv
digestion_isoforms_Trypsin_filtered.to_csv(os.path.join(output_dir, 'digestion_isoforms_Trypsin_filtered.csv'), index=False)
digestion_isoforms_TrypsinP_filtered.to_csv(os.path.join(output_dir, 'digestion_isoforms_TrypsinP_filtered.csv'), index=False)